# 实验6：防御策略实现（BNP 剪枝）

本实验面向 **实验4：BadNets 后门攻击实现** 产生的带后门模型，使用 BNP（Batch-Normalization/Normalization Pruning）思路进行防御：先加载实验4现场训练保存的 `demo_last.ckpt`，再通过对比干净样本与加 Trigger 样本在归一化层通道上的激活差异，定位对 Trigger 异常敏感的通道，并把这些通道的归一化缩放参数置零，从而削弱或移除后门神经元。

## 一、使用说明

推荐流程：

1. 先从项目根目录运行 `04_后门攻击实现_BadNets.ipynb`，完成 square 和/或 checkerboard 后门模型训练。
2. 再打开本 notebook，按顺序运行全部单元。
3. 重点观察 BNP 防御前后的三类指标：
   - `clean_accuracy_before / clean_accuracy_after`：正常样本准确率是否尽量保持；
   - `ASR_before / ASR_after`：触发器攻击成功率是否下降；
   - `pruned_channel_count`：BNP 移除的异常通道数量。

本项目当前实验4默认使用 `norm_type="group"` 训练模型。BNP 的核心对象原本是 BatchNorm 通道；为了严格加载实验4实际生成的模型，本 notebook 默认自动识别 checkpoint，如果是 GroupNorm，则使用同样的“按通道归一化缩放参数剪枝”兼容执行。若你把实验4改为 `norm_type="batch"` 重新训练，本 notebook 会自动按真正的 BatchNorm 层执行 BNP。

**来源声明：本 notebook 基于 `04_后门攻击实现_BadNets.ipynb` 生成；运行时仅读取项目目录和实验4产生的后门 checkpoint。**


## 二、环境检查与运行初始化

按顺序运行下面的代码单元，完成 CANN 路径设置、TBE 连通性检查、依赖导入以及 MindSpore/Ascend 配置。云上 CANN 路径不同时，只需修改 `cann_root`；本实验默认使用 Ascend/NPU。

<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
table th,
table td {
  text-align: left !important;
}
</style>


In [ ]:
import os
import sys

cann_root = "/home/developer/Ascend/cann-9.0.0"
cann_python = f"{cann_root}/python/site-packages"

os.environ["ASCEND_HOME_PATH"] = cann_root
os.environ["ASCEND_OPP_PATH"] = f"{cann_root}/opp"
os.environ["PYTHONPATH"] = cann_python + ":" + os.environ.get("PYTHONPATH", "")
os.environ["LD_LIBRARY_PATH"] = (
    f"{cann_root}/lib64:"
    f"{cann_root}/runtime/lib64:"
    f"{cann_root}/tools/aoe/lib64:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

if cann_python not in sys.path:
    sys.path.insert(0, cann_python)

print("CANN env ready")
print(sys.executable)
import csv
import json
import math
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("GLOG_v", "3")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

RUN_PROFILE = os.environ.get("ONSITE_DEMO_PROFILE", "cloud_live").strip().lower()
PREFERRED_DEVICE = os.environ.get("BNP_DEVICE_TARGET", "Ascend").strip()
MS_MODE = os.environ.get("BNP_MS_MODE", "PYNATIVE").strip().upper()
ALLOW_CPU_FALLBACK = os.environ.get("BNP_ALLOW_CPU_FALLBACK", "0").strip() == "1"

# 需要防御的实验4触发器。默认两条都处理；若只想跑 checkerboard，可设置 BNP_TRIGGER_TYPES=checkerboard。
DEFENSE_TRIGGER_TYPES = [
    item.strip().lower()
    for item in os.environ.get("BNP_TRIGGER_TYPES", "square,checkerboard").split(",")
    if item.strip()
]

# auto 会从 checkpoint 参数名中识别 batch/group；也可显式设置为 batch 或 group。
BNP_MODEL_NORM_TYPE = os.environ.get("BNP_MODEL_NORM_TYPE", "auto").strip().lower()

# BNP 剪枝超参数。ratio 是全局通道比例；max layer ratio 避免某一层被一次性剪太多。
PRUNE_RATIOS = [float(x) for x in os.environ.get("BNP_PRUNE_RATIOS", "0.01,0.03,0.05,0.10").split(",")]
BNP_MAX_LAYER_PRUNE_RATIO = float(os.environ.get("BNP_MAX_LAYER_PRUNE_RATIO", "0.25"))
BNP_MIN_ANOMALY_SCORE = float(os.environ.get("BNP_MIN_ANOMALY_SCORE", "0.0"))
BNP_CLEAN_DROP_PENALTY = float(os.environ.get("BNP_CLEAN_DROP_PENALTY", "1.0"))
CALIBRATION_MAX_SAMPLES = int(os.environ.get("BNP_CALIBRATION_MAX_SAMPLES", "96"))
EVAL_BATCH_SIZE = int(os.environ.get("BNP_EVAL_BATCH_SIZE", "1"))
SEED_OFFSET = int(os.environ.get("BNP_SEED_OFFSET", "2024"))

# 严格模式：默认不使用官方 fallback checkpoint，确保模型来自实验4现场运行输出。
ALLOW_OFFICIAL_POISONED_FALLBACK = os.environ.get("BNP_ALLOW_OFFICIAL_POISONED_FALLBACK", "0").strip() == "1"

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "paths.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break

from demo_lib.evaluation import evaluate_small_subset
from demo_lib.inference import (
    apply_optional_trigger,
    load_image_chw_01,
    load_model_from_checkpoint,
    normalize_for_model,
    predict_one_image,
)
from demo_lib.paths import (
    ensure_dir,
    find_checkpoint,
    load_demo_config,
    make_run_timestamp,
    resolve_demo_mode_settings,
    resolve_project_root,
    save_json,
    stage_output_name,
)
from demo_lib.runtime import configure_mindspore_device
from demo_lib.subset import create_demo_subset, list_image_records, pick_first_non_target_image

paths = resolve_project_root()
PROJECT_ROOT = paths["PROJECT_ROOT"]
CONFIG = load_demo_config()
MODE_SETTINGS = resolve_demo_mode_settings(CONFIG, RUN_PROFILE)
TARGET_LABEL = int(CONFIG["target_label"])
SEED = int(CONFIG.get("seed", 42)) + SEED_OFFSET
DEFENSE_RUN_ROOT = ensure_dir(paths["DEMO_RUNS_ROOT"] / f"bnp_defense_{make_run_timestamp()}")

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"run profile = {RUN_PROFILE}")
print(f"device target preference = {PREFERRED_DEVICE}")
print(f"allow CPU fallback = {ALLOW_CPU_FALLBACK}")
print(f"MindSpore mode = {MS_MODE}")
print(f"target label = {TARGET_LABEL}")
print(f"defense trigger types = {DEFENSE_TRIGGER_TYPES}")
print(f"norm type setting = {BNP_MODEL_NORM_TYPE}")
print(f"prune ratios = {PRUNE_RATIOS}")
print(f"defense output root = {DEFENSE_RUN_ROOT}")


def display_table(rows):
    try:
        import pandas as pd

        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(json.dumps(row, ensure_ascii=False, indent=2))


def save_rows_csv(rows, path):
    path = Path(path)
    ensure_dir(path.parent)
    if not rows:
        path.write_text("", encoding="utf-8")
        return path
    fieldnames = list(rows[0].keys())
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    return path
try:
    import mindspore as ms
except Exception as exc:
    raise RuntimeError("MindSpore 不可导入，请确认当前 Notebook 使用 MindSpore/Ascend 环境。") from exc

ACTUAL_DEVICE = configure_mindspore_device(
    preferred=PREFERRED_DEVICE,
    ms_mode=MS_MODE,
    allow_cpu_fallback=ALLOW_CPU_FALLBACK,
)

print(f"MindSpore version = {ms.__version__}")
print(f"actual device target = {ACTUAL_DEVICE}")

## 三、定位实验4产生的后门模型

本单元会在 `onsite_demo/outputs/demo_runs/notebook_*/` 下寻找实验4输出的 `demo_train_log.json` 和 `demo_last.ckpt`。这一步是本实验的关键：防御对象必须是实验4 BadNets notebook 生成的带后门模型。


In [ ]:
def _load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _resolve_logged_path(raw_path, base_dir):
    if not raw_path:
        return None
    path = Path(str(raw_path))
    if path.is_absolute():
        return path
    candidates = [PROJECT_ROOT / path, Path(base_dir) / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def locate_badnets_artifact(trigger_type):
    trigger_type = str(trigger_type).lower().strip()
    stage_dir_name = stage_output_name(trigger_type)
    experiment_name = str(CONFIG[trigger_type]["experiment_name"])

    log_candidates = sorted(
        paths["DEMO_RUNS_ROOT"].glob(f"*/{stage_dir_name}/demo_train_log.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    checked = []
    for log_path in log_candidates:
        stage_dir = log_path.parent
        run_root = stage_dir.parent
        if not run_root.name.startswith("notebook_"):
            # 只接受实验4 notebook 的现场训练目录，避免误把防御输出当成攻击模型。
            continue
        log = _load_json(log_path)
        ckpt_path = _resolve_logged_path(log.get("demo_checkpoint_path"), stage_dir)
        if ckpt_path is None or not ckpt_path.exists():
            fallback_ckpt = stage_dir / "demo_last.ckpt"
            ckpt_path = fallback_ckpt if fallback_ckpt.exists() else ckpt_path
        checked.append(str(log_path))
        if ckpt_path is not None and ckpt_path.exists():
            eval_path = stage_dir / "demo_eval_summary.json"
            return {
                "trigger_type": trigger_type,
                "experiment_name": experiment_name,
                "stage_dir": str(stage_dir),
                "run_root": str(run_root),
                "train_log_path": str(log_path),
                "eval_summary_path": str(eval_path) if eval_path.exists() else "",
                "checkpoint_path": str(ckpt_path),
                "model_source": "experiment4_live_demo_last_ckpt",
                "epochs_completed": int(log.get("epochs_completed", 0)),
                "final_clean_accuracy_from_attack_log": log.get("final_clean_accuracy"),
                "final_asr_from_attack_log": log.get("final_asr"),
            }

    if ALLOW_OFFICIAL_POISONED_FALLBACK:
        ckpt_path = find_checkpoint(experiment_name)
        return {
            "trigger_type": trigger_type,
            "experiment_name": experiment_name,
            "stage_dir": "",
            "run_root": "",
            "train_log_path": "",
            "eval_summary_path": "",
            "checkpoint_path": str(ckpt_path),
            "model_source": "official_poisoned_badnets_checkpoint_fallback",
            "epochs_completed": None,
            "final_clean_accuracy_from_attack_log": None,
            "final_asr_from_attack_log": None,
        }

    detail = "\n".join(checked[:8]) if checked else "未找到任何实验4现场训练日志。"
    raise FileNotFoundError(
        f"未找到 trigger={trigger_type} 的实验4现场后门模型 demo_last.ckpt。\n"
        f"请先从项目根目录运行 04_后门攻击实现_BadNets.ipynb。\n"
        f"已检查：\n{detail}"
    )


badnets_artifacts = []
for trigger_type in DEFENSE_TRIGGER_TYPES:
    try:
        badnets_artifacts.append(locate_badnets_artifact(trigger_type))
    except Exception as exc:
        print(f"跳过 {trigger_type}: {exc}")

if not badnets_artifacts:
    raise RuntimeError(
        "没有可防御的实验4后门模型。请先运行 04_后门攻击实现_BadNets.ipynb，"
        "确保生成 onsite_demo/outputs/demo_runs/notebook_*/.../demo_last.ckpt。"
    )

display_table(badnets_artifacts)


#### 讲解：为什么不直接使用干净模型

BNP 防御的目标是恢复已经被 BadNets 植入后门的模型，而不是重新训练一个干净模型。因此上一步强制查找实验4输出的 `demo_last.ckpt`。如果你看到 `model_source=experiment4_live_demo_last_ckpt`，说明防御对象满足要求。


## 四、BNP：异常通道定位与剪枝工具函数

本实现采用“触发器敏感度”作为 BNP 通道异常分数：

\[
score_c = \max(0, \frac{E|a_c(x_{trigger})| - E|a_c(x_{clean})|}{Std(|a_c(x_{clean})|)+\epsilon})
\]

分数越高，表示该通道对 Trigger 的激活提升越异常。剪枝时对分数最高的通道执行归一化层通道门控，即把对应通道的 `gamma` 和 `beta` 置零。


In [3]:
def infer_norm_type_from_checkpoint(checkpoint_path):
    if BNP_MODEL_NORM_TYPE in {"batch", "group", "none"}:
        return BNP_MODEL_NORM_TYPE
    param_dict = ms.load_checkpoint(str(checkpoint_path))
    keys = list(param_dict.keys())
    if any(("moving_mean" in key or "moving_variance" in key) for key in keys):
        return "batch"
    return "group"


def _child_sort_key(item):
    name, _ = item
    text = str(name)
    return (0, int(text)) if text.isdigit() else (1, text)


def named_children(cell):
    try:
        items = list(cell.name_cells().items())
    except Exception:
        return []
    filtered = [(name, child) for name, child in items if child is not cell]
    return sorted(filtered, key=_child_sort_key)


def is_supported_norm_layer(cell):
    cls_name = cell.__class__.__name__.lower()
    return any(key in cls_name for key in ("batchnorm", "groupnorm"))


def collect_feature_norm_layers(model):
    """Collect feature-space normalization layers that have channel-wise gamma/beta."""
    layers = {}
    features = getattr(model, "features", None)
    if features is None:
        return layers
    for feature_name, feature_cell in named_children(features):
        block = getattr(feature_cell, "block", None)
        if block is None:
            continue
        for child_name, child_cell in named_children(block):
            if is_supported_norm_layer(child_cell):
                layers[f"features.{feature_name}.block.{child_name}"] = child_cell
    return layers


def forward_collect_feature_norms(model, input_tensor):
    """Run only model.features and collect outputs immediately after supported norm layers."""
    activations = {}
    features = getattr(model, "features", None)
    if features is None:
        raise AttributeError("当前模型没有 features，无法执行 BNP 激活统计。")

    x = input_tensor
    for feature_name, feature_cell in named_children(features):
        block = getattr(feature_cell, "block", None)
        if block is not None:
            for child_name, child_cell in named_children(block):
                x = child_cell(x)
                if is_supported_norm_layer(child_cell):
                    activations[f"features.{feature_name}.block.{child_name}"] = x.asnumpy()
        else:
            x = feature_cell(x)
    return activations


def per_sample_channel_abs_mean(activation):
    arr = np.asarray(activation, dtype=np.float32)
    if arr.ndim < 2:
        raise ValueError(f"activation ndim must be >= 2, got {arr.shape}")
    if arr.ndim == 2:
        return np.abs(arr)
    spatial_axes = tuple(axis for axis in range(arr.ndim) if axis not in (0, 1))
    return np.mean(np.abs(arr), axis=spatial_axes)


def compute_bnp_channel_scores(
    model,
    subset_test_dir,
    trigger_type,
    target_label,
    max_samples=96,
    batch_size=1,
    trigger_size=4,
    alpha=0.8,
    position="bottom_right",
    seed=42,
):
    records = [(path, label) for path, label in list_image_records(subset_test_dir) if int(label) != int(target_label)]
    if not records:
        raise ValueError(f"校准集没有非目标类样本，无法计算 ASR/BNP 分数：{subset_test_dir}")
    rng = np.random.default_rng(int(seed))
    indices = np.arange(len(records))
    rng.shuffle(indices)
    records = [records[int(i)] for i in indices[: int(max_samples)]]

    model.set_train(False)
    accumulator = {}
    sample_count = 0
    for start in range(0, len(records), int(batch_size)):
        batch_records = records[start : start + int(batch_size)]
        clean_images = np.stack([load_image_chw_01(path) for path, _ in batch_records]).astype(np.float32)
        triggered_images = np.stack([
            apply_optional_trigger(
                image,
                trigger_type=trigger_type,
                trigger_size=trigger_size,
                alpha=alpha,
                position=position,
            )
            for image in clean_images
        ]).astype(np.float32)

        clean_tensor = ms.Tensor(normalize_for_model(clean_images), ms.float32)
        triggered_tensor = ms.Tensor(normalize_for_model(triggered_images), ms.float32)
        clean_acts = forward_collect_feature_norms(model, clean_tensor)
        triggered_acts = forward_collect_feature_norms(model, triggered_tensor)

        for layer_name, clean_activation in clean_acts.items():
            if layer_name not in triggered_acts:
                continue
            clean_stat = per_sample_channel_abs_mean(clean_activation).astype(np.float64)
            triggered_stat = per_sample_channel_abs_mean(triggered_acts[layer_name]).astype(np.float64)
            n = int(clean_stat.shape[0])
            if layer_name not in accumulator:
                channels = int(clean_stat.shape[1])
                accumulator[layer_name] = {
                    "n": 0,
                    "sum_clean": np.zeros(channels, dtype=np.float64),
                    "sum_clean_sq": np.zeros(channels, dtype=np.float64),
                    "sum_triggered": np.zeros(channels, dtype=np.float64),
                }
            accumulator[layer_name]["n"] += n
            accumulator[layer_name]["sum_clean"] += clean_stat.sum(axis=0)
            accumulator[layer_name]["sum_clean_sq"] += np.square(clean_stat).sum(axis=0)
            accumulator[layer_name]["sum_triggered"] += triggered_stat.sum(axis=0)
        sample_count += len(batch_records)

    layer_scores = {}
    flat_rows = []
    for layer_name, item in accumulator.items():
        n = max(int(item["n"]), 1)
        clean_mean = item["sum_clean"] / n
        triggered_mean = item["sum_triggered"] / n
        clean_var = np.maximum(item["sum_clean_sq"] / n - np.square(clean_mean), 0.0)
        clean_std = np.sqrt(clean_var)
        lift = triggered_mean - clean_mean
        score = np.maximum(lift, 0.0) / (clean_std + 1e-6)
        layer_scores[layer_name] = {
            "score": score,
            "clean_mean": clean_mean,
            "triggered_mean": triggered_mean,
            "lift": lift,
            "clean_std": clean_std,
        }
        for channel_idx in range(len(score)):
            flat_rows.append({
                "layer": layer_name,
                "channel": int(channel_idx),
                "score": float(score[channel_idx]),
                "clean_mean_abs_activation": float(clean_mean[channel_idx]),
                "triggered_mean_abs_activation": float(triggered_mean[channel_idx]),
                "activation_lift": float(lift[channel_idx]),
                "clean_std": float(clean_std[channel_idx]),
            })

    flat_rows = sorted(flat_rows, key=lambda row: row["score"], reverse=True)
    return layer_scores, flat_rows, sample_count


def snapshot_norm_parameters(layer_cells):
    snapshot = {}
    for layer_name, cell in layer_cells.items():
        snapshot[layer_name] = {}
        for attr in ("gamma", "beta"):
            param = getattr(cell, attr, None)
            if param is not None:
                snapshot[layer_name][attr] = param.asnumpy().copy()
    return snapshot


def restore_norm_parameters(layer_cells, snapshot):
    for layer_name, params in snapshot.items():
        cell = layer_cells[layer_name]
        for attr, array in params.items():
            param = getattr(cell, attr, None)
            if param is not None:
                param.set_data(ms.Tensor(array.copy(), param.dtype))


def select_prune_channels(layer_scores, prune_ratio, max_layer_ratio=0.25, min_score=0.0):
    total_channels = sum(len(item["score"]) for item in layer_scores.values())
    target_k = max(1, int(math.ceil(float(prune_ratio) * max(total_channels, 1))))
    candidates = []
    for layer_name, item in layer_scores.items():
        scores = item["score"]
        for channel_idx, score in enumerate(scores):
            if float(score) > float(min_score):
                candidates.append((float(score), layer_name, int(channel_idx)))
    candidates.sort(reverse=True)

    selected = {}
    layer_counts = {}
    for score, layer_name, channel_idx in candidates:
        layer_size = len(layer_scores[layer_name]["score"])
        layer_cap = max(1, int(math.ceil(float(max_layer_ratio) * layer_size)))
        if layer_counts.get(layer_name, 0) >= layer_cap:
            continue
        selected.setdefault(layer_name, []).append(channel_idx)
        layer_counts[layer_name] = layer_counts.get(layer_name, 0) + 1
        if sum(len(v) for v in selected.values()) >= target_k:
            break

    selected_rows = []
    for layer_name, channels in selected.items():
        item = layer_scores[layer_name]
        for channel_idx in sorted(channels):
            selected_rows.append({
                "layer": layer_name,
                "channel": int(channel_idx),
                "score": float(item["score"][channel_idx]),
                "clean_mean_abs_activation": float(item["clean_mean"][channel_idx]),
                "triggered_mean_abs_activation": float(item["triggered_mean"][channel_idx]),
                "activation_lift": float(item["lift"][channel_idx]),
            })
    selected_rows.sort(key=lambda row: row["score"], reverse=True)
    return selected, selected_rows


def apply_bnp_pruning(layer_cells, selected):
    pruned_count = 0
    for layer_name, channels in selected.items():
        if layer_name not in layer_cells:
            continue
        cell = layer_cells[layer_name]
        channel_indices = np.asarray(sorted(set(int(c) for c in channels)), dtype=np.int64)
        for attr in ("gamma", "beta"):
            param = getattr(cell, attr, None)
            if param is None:
                continue
            array = param.asnumpy().copy()
            valid = channel_indices[(channel_indices >= 0) & (channel_indices < array.shape[0])]
            if len(valid) == 0:
                continue
            array[valid] = 0.0
            param.set_data(ms.Tensor(array, param.dtype))
        pruned_count += int(len(channel_indices))
    return pruned_count


## 五、加载后门模型并执行 BNP 防御

每个触发器会执行完整流程：

1. 加载实验4输出的后门 checkpoint；
2. 在剪枝前评估 clean accuracy 与 ASR；
3. 用干净/触发样本激活差异计算 BNP 异常分数；
4. 扫描多个剪枝比例，选择“ASR 降低多、clean accuracy 损失小”的候选；
5. 保存 BNP 防御后的 checkpoint 与指标报告。


In [ ]:
def resolve_eval_subset_for_artifact(artifact):
    run_root = Path(str(artifact.get("run_root", ""))) if artifact.get("run_root") else None
    if run_root is not None:
        candidate = run_root / "demo_subset" / "test"
        if candidate.exists():
            return candidate

    # 如果实验4输出目录中没有 demo_subset，仅为评估/校准重建一个 subset；模型仍然来自实验4 ckpt。
    subset_root = ensure_dir(DEFENSE_RUN_ROOT / f"eval_subset_{artifact['trigger_type']}")
    if not (subset_root / "test").exists():
        create_demo_subset(
            train_dir=paths["TRAIN_DIR"],
            test_dir=paths["TEST_DIR"],
            output_dir=subset_root,
            train_per_class=int(MODE_SETTINGS["train_per_class"]),
            test_per_class=int(MODE_SETTINGS["test_per_class"]),
            seed=SEED,
        )
    return subset_root / "test"


def compact_eval_row(prefix, summary):
    return {
        f"clean_accuracy_{prefix}": round(float(summary["clean_accuracy"]), 4),
        f"ASR_{prefix}": round(float(summary["attack_success_rate"]), 4),
        f"target_confidence_{prefix}": round(float(summary.get("avg_target_confidence_on_triggered", 0.0)), 4),
    }


all_summary_rows = []
all_sweep_rows = []
defense_reports = []

for artifact in badnets_artifacts:
    trigger_type = artifact["trigger_type"]
    stage_cfg = CONFIG[trigger_type]
    experiment_name = artifact["experiment_name"]
    checkpoint_path = Path(artifact["checkpoint_path"])
    output_dir = ensure_dir(DEFENSE_RUN_ROOT / stage_output_name(trigger_type))
    eval_subset_dir = resolve_eval_subset_for_artifact(artifact)
    inferred_norm_type = infer_norm_type_from_checkpoint(checkpoint_path)

    print("=" * 88)
    print(f"BNP defense target trigger = {trigger_type}")
    print(f"loading BadNets checkpoint from experiment4 = {checkpoint_path}")
    print(f"model source = {artifact['model_source']}")
    print(f"inferred norm type = {inferred_norm_type}")
    print(f"eval/calibration subset = {eval_subset_dir}")

    model = load_model_from_checkpoint(
        experiment_name=experiment_name,
        checkpoint_path=checkpoint_path,
        num_classes=43,
        norm_type=inferred_norm_type,
        device_target=ACTUAL_DEVICE,
        ms_mode=MS_MODE,
        allow_cpu_fallback=ALLOW_CPU_FALLBACK,
    )
    model.set_train(False)

    layer_cells = collect_feature_norm_layers(model)
    if not layer_cells:
        raise RuntimeError(
            f"模型中没有找到可剪枝的 BatchNorm/GroupNorm 特征层。"
            f"请确认 checkpoint 与 norm_type={inferred_norm_type} 匹配。"
        )
    layer_rows = [{"layer": name, "layer_type": cell.__class__.__name__} for name, cell in layer_cells.items()]
    print("BNP candidate norm layers:")
    display_table(layer_rows)

    before_eval = evaluate_small_subset(
        model=model,
        subset_test_dir=eval_subset_dir,
        trigger_type=trigger_type,
        target_label=TARGET_LABEL,
        trigger_size=int(stage_cfg.get("trigger_size", 4)),
        alpha=float(stage_cfg.get("alpha", 0.8)),
        position=str(stage_cfg.get("position", "bottom_right")),
        batch_size=EVAL_BATCH_SIZE,
        output_path=output_dir / "before_bnp_eval_summary.json",
    )

    layer_scores, score_rows, calibration_count = compute_bnp_channel_scores(
        model=model,
        subset_test_dir=eval_subset_dir,
        trigger_type=trigger_type,
        target_label=TARGET_LABEL,
        max_samples=CALIBRATION_MAX_SAMPLES,
        batch_size=EVAL_BATCH_SIZE,
        trigger_size=int(stage_cfg.get("trigger_size", 4)),
        alpha=float(stage_cfg.get("alpha", 0.8)),
        position=str(stage_cfg.get("position", "bottom_right")),
        seed=SEED,
    )
    score_csv = save_rows_csv(score_rows, output_dir / "bnp_channel_scores.csv")
    print(f"calibration sample count = {calibration_count}")
    print(f"BNP channel score CSV = {score_csv}")
    print("Top abnormal channels:")
    display_table(score_rows[:12])

    snapshot = snapshot_norm_parameters(layer_cells)
    sweep_rows = []
    best_item = None

    for prune_ratio in PRUNE_RATIOS:
        restore_norm_parameters(layer_cells, snapshot)
        selected, selected_rows = select_prune_channels(
            layer_scores=layer_scores,
            prune_ratio=prune_ratio,
            max_layer_ratio=BNP_MAX_LAYER_PRUNE_RATIO,
            min_score=BNP_MIN_ANOMALY_SCORE,
        )
        pruned_count = apply_bnp_pruning(layer_cells, selected)
        selected_csv = save_rows_csv(
            selected_rows,
            output_dir / f"bnp_selected_channels_ratio_{prune_ratio:.3f}.csv",
        )
        pruned_ckpt_path = output_dir / f"bnp_pruned_ratio_{prune_ratio:.3f}.ckpt"
        ms.save_checkpoint(model, str(pruned_ckpt_path))

        after_eval = evaluate_small_subset(
            model=model,
            subset_test_dir=eval_subset_dir,
            trigger_type=trigger_type,
            target_label=TARGET_LABEL,
            trigger_size=int(stage_cfg.get("trigger_size", 4)),
            alpha=float(stage_cfg.get("alpha", 0.8)),
            position=str(stage_cfg.get("position", "bottom_right")),
            batch_size=EVAL_BATCH_SIZE,
            output_path=output_dir / f"after_bnp_eval_ratio_{prune_ratio:.3f}.json",
        )
        clean_drop = float(before_eval["clean_accuracy"]) - float(after_eval["clean_accuracy"])
        asr_drop = float(before_eval["attack_success_rate"]) - float(after_eval["attack_success_rate"])
        selection_score = asr_drop - BNP_CLEAN_DROP_PENALTY * max(clean_drop, 0.0)
        row = {
            "trigger_type": trigger_type,
            "model_source": artifact["model_source"],
            "norm_type": inferred_norm_type,
            "prune_ratio": float(prune_ratio),
            "pruned_channel_count": int(pruned_count),
            "calibration_sample_count": int(calibration_count),
            "clean_accuracy_before": float(before_eval["clean_accuracy"]),
            "ASR_before": float(before_eval["attack_success_rate"]),
            "clean_accuracy_after": float(after_eval["clean_accuracy"]),
            "ASR_after": float(after_eval["attack_success_rate"]),
            "clean_accuracy_drop": float(clean_drop),
            "ASR_drop": float(asr_drop),
            "selection_score": float(selection_score),
            "checkpoint_path": str(pruned_ckpt_path),
            "selected_channels_csv": str(selected_csv),
        }
        sweep_rows.append(row)
        all_sweep_rows.append(row)
        if best_item is None or selection_score > best_item["selection_score"]:
            best_item = {**row, "selected": selected, "selected_rows": selected_rows}

    if best_item is None:
        raise RuntimeError(f"trigger={trigger_type} 没有产生有效 BNP 剪枝候选。")

    # 恢复原始后门模型参数后，只应用最佳候选，保存最终防御模型。
    restore_norm_parameters(layer_cells, snapshot)
    apply_bnp_pruning(layer_cells, best_item["selected"])
    final_ckpt_path = output_dir / "bnp_defended_best.ckpt"
    ms.save_checkpoint(model, str(final_ckpt_path))
    final_eval = evaluate_small_subset(
        model=model,
        subset_test_dir=eval_subset_dir,
        trigger_type=trigger_type,
        target_label=TARGET_LABEL,
        trigger_size=int(stage_cfg.get("trigger_size", 4)),
        alpha=float(stage_cfg.get("alpha", 0.8)),
        position=str(stage_cfg.get("position", "bottom_right")),
        batch_size=EVAL_BATCH_SIZE,
        output_path=output_dir / "after_bnp_best_eval_summary.json",
    )

    sweep_csv = save_rows_csv(sweep_rows, output_dir / "bnp_sweep_results.csv")
    final_selected_csv = save_rows_csv(best_item["selected_rows"], output_dir / "bnp_best_selected_channels.csv")
    report = {
        "trigger_type": trigger_type,
        "experiment_name": experiment_name,
        "model_source": artifact["model_source"],
        "badnets_checkpoint_path": str(checkpoint_path),
        "norm_type": inferred_norm_type,
        "eval_subset_dir": str(eval_subset_dir),
        "target_label": TARGET_LABEL,
        "calibration_sample_count": calibration_count,
        "before_eval": before_eval,
        "best_prune_ratio": best_item["prune_ratio"],
        "best_pruned_channel_count": best_item["pruned_channel_count"],
        "best_selection_score": best_item["selection_score"],
        "after_best_eval": final_eval,
        "bnp_channel_scores_csv": str(score_csv),
        "bnp_sweep_results_csv": str(sweep_csv),
        "bnp_best_selected_channels_csv": str(final_selected_csv),
        "bnp_defended_checkpoint_path": str(final_ckpt_path),
    }
    save_json(report, output_dir / "bnp_defense_report.json")
    defense_reports.append(report)

    summary_row = {
        "trigger_type": trigger_type,
        "model_source": artifact["model_source"],
        "norm_type": inferred_norm_type,
        "badnets_checkpoint_path": str(checkpoint_path),
        "best_prune_ratio": float(best_item["prune_ratio"]),
        "best_pruned_channel_count": int(best_item["pruned_channel_count"]),
        **compact_eval_row("before", before_eval),
        **compact_eval_row("after", final_eval),
        "defended_checkpoint_path": str(final_ckpt_path),
        "report_path": str(output_dir / "bnp_defense_report.json"),
    }
    all_summary_rows.append(summary_row)

summary_csv = save_rows_csv(all_summary_rows, DEFENSE_RUN_ROOT / "bnp_defense_summary.csv")
sweep_csv_all = save_rows_csv(all_sweep_rows, DEFENSE_RUN_ROOT / "bnp_defense_all_sweep_results.csv")
save_json(defense_reports, DEFENSE_RUN_ROOT / "bnp_defense_reports.json")

print("BNP 防御总览：")
display_table(all_summary_rows)
print(f"summary csv = {summary_csv}")
print(f"all sweep csv = {sweep_csv_all}")
print(f"all reports json = {DEFENSE_RUN_ROOT / 'bnp_defense_reports.json'}")


## 六、防御效果可视化

理想结果是：BNP 后 `ASR_after` 明显低于 `ASR_before`，同时 `clean_accuracy_after` 与 `clean_accuracy_before` 接近。若 clean accuracy 掉得过多，可以降低 `BNP_PRUNE_RATIOS` 或 `BNP_MAX_LAYER_PRUNE_RATIO` 后重新运行。


In [ ]:
if not all_sweep_rows:
    raise RuntimeError("没有可视化的 BNP sweep 结果。")

try:
    import pandas as pd
    sweep_df = pd.DataFrame(all_sweep_rows)
    display(sweep_df)
except Exception:
    sweep_df = None
    display_table(all_sweep_rows)

for trigger_type in sorted(set(row["trigger_type"] for row in all_sweep_rows)):
    rows = [row for row in all_sweep_rows if row["trigger_type"] == trigger_type]
    rows = sorted(rows, key=lambda row: row["prune_ratio"])
    x = [row["prune_ratio"] for row in rows]
    clean_values = [row["clean_accuracy_after"] for row in rows]
    asr_values = [row["ASR_after"] for row in rows]
    before_clean = rows[0]["clean_accuracy_before"]
    before_asr = rows[0]["ASR_before"]

    fig, ax = plt.subplots(figsize=(7.2, 4.0))
    ax.plot(x, clean_values, marker="o", label="Clean Accuracy after BNP")
    ax.plot(x, asr_values, marker="o", label="ASR after BNP")
    ax.axhline(before_clean, linestyle="--", linewidth=1.2, label="Clean Accuracy before BNP")
    ax.axhline(before_asr, linestyle="--", linewidth=1.2, label="ASR before BNP")
    ax.set_xlabel("BNP prune ratio")
    ax.set_ylabel("Score")
    ax.set_ylim(0.0, 1.05)
    ax.set_title(f"BNP Defense Sweep - {trigger_type}")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    display(fig)
    plt.close(fig)


## 七、单样本防御前后对比

本单元选择一个非目标类测试样本，比较 BNP 前后模型在 clean / triggered 输入上的预测。若防御有效，`triggered` 输入不应再稳定预测为目标类 `target_label=0`。


In [ ]:
def show_prediction_rows(model, sample_image, trigger_type, target_label, title):
    clean_result = predict_one_image(model, sample_image, trigger_type=None, target_label=target_label)
    triggered_result = predict_one_image(
        model,
        sample_image,
        trigger_type=trigger_type,
        target_label=target_label,
        trigger_size=int(CONFIG[trigger_type].get("trigger_size", 4)),
        alpha=float(CONFIG[trigger_type].get("alpha", 0.8)),
        position=str(CONFIG[trigger_type].get("position", "bottom_right")),
    )
    rows = []
    for name, result in [("clean", clean_result), ("triggered", triggered_result)]:
        rows.append({
            "title": title,
            "input_type": name,
            "pred_label": int(result["pred_label"]),
            "pred_probability": round(float(result["probability"]), 4),
            "target_label": int(target_label),
            "target_probability": round(float(result.get("target_probability", 0.0)), 4),
            "attack_success": bool(name == "triggered" and int(result["pred_label"]) == int(target_label)),
        })
    return rows, clean_result, triggered_result


sample_rows = []
for report in defense_reports:
    trigger_type = report["trigger_type"]
    eval_subset_dir = Path(report["eval_subset_dir"])
    sample_image = pick_first_non_target_image(eval_subset_dir, target_label=TARGET_LABEL)

    before_model = load_model_from_checkpoint(
        experiment_name=report["experiment_name"],
        checkpoint_path=report["badnets_checkpoint_path"],
        num_classes=43,
        norm_type=report["norm_type"],
        device_target=ACTUAL_DEVICE,
        ms_mode=MS_MODE,
        allow_cpu_fallback=ALLOW_CPU_FALLBACK,
    )
    after_model = load_model_from_checkpoint(
        experiment_name=report["experiment_name"],
        checkpoint_path=report["bnp_defended_checkpoint_path"],
        num_classes=43,
        norm_type=report["norm_type"],
        device_target=ACTUAL_DEVICE,
        ms_mode=MS_MODE,
        allow_cpu_fallback=ALLOW_CPU_FALLBACK,
    )

    before_rows, before_clean, before_triggered = show_prediction_rows(
        before_model, sample_image, trigger_type, TARGET_LABEL, f"{trigger_type} before BNP"
    )
    after_rows, after_clean, after_triggered = show_prediction_rows(
        after_model, sample_image, trigger_type, TARGET_LABEL, f"{trigger_type} after BNP"
    )
    sample_rows.extend(before_rows + after_rows)

    fig, axes = plt.subplots(1, 4, figsize=(11.2, 3.0))
    images_and_titles = [
        (before_clean["processed_image"], f"Before clean\npred={before_clean['pred_label']}"),
        (before_triggered["processed_image"], f"Before triggered\npred={before_triggered['pred_label']}"),
        (after_clean["processed_image"], f"After clean\npred={after_clean['pred_label']}"),
        (after_triggered["processed_image"], f"After triggered\npred={after_triggered['pred_label']}"),
    ]
    for ax, (image, title) in zip(axes, images_and_titles):
        ax.imshow(np.clip(image, 0.0, 1.0))
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"Single Sample BNP Defense - {trigger_type}")
    plt.tight_layout()
    display(fig)
    plt.close(fig)

display_table(sample_rows)


## 八、实验6结果解读

BNP 防御的核心判断标准不是“剪得越多越好”，而是在后门攻击成功率和正常准确率之间做平衡：

- 如果 `ASR_after` 大幅下降且 `clean_accuracy_after` 基本保持，说明 Trigger 相关异常通道被有效抑制；
- 如果 `ASR_after` 下降不明显，可以提高 `BNP_PRUNE_RATIOS` 的上限或增加 `BNP_CALIBRATION_MAX_SAMPLES`；
- 如果 `clean_accuracy_after` 掉得太多，应降低剪枝比例，或减小 `BNP_MAX_LAYER_PRUNE_RATIO`；
- 若当前 checkpoint 使用 GroupNorm，本实验输出会显示 `norm_type=group`，这是为了严格加载实验4生成的后门模型而做的通道剪枝兼容路径；若需要严格意义的 BatchNorm BNP，请先在实验4中用 `norm_type="batch"` 重新生成后门模型，再运行本 notebook。

## 结论

本 notebook 完成了面向 BadNets 后门模型的 BNP 防御流程：严格加载实验4产生的带后门 checkpoint，通过 clean/triggered 激活差异定位归一化层异常通道，剪枝后保存防御 checkpoint，并用 clean accuracy 与 ASR 验证模型鲁棒性恢复效果。
